# Tutorial 1: Your First Scan

In this tutorial, you'll learn how to:
- Create your first error catalog
- Run a basic scan on local files
- View and understand the results
- Export reports in different formats

## Prerequisites

- `autosubmit-scan` installed (see [Installation Guide](../getting-started/installation))
- Basic familiarity with command-line interfaces
- Some log files to scan (we'll create sample files)

## Setup: Creating Sample Log Files

First, let's create a directory with sample log files that contain various error patterns.

In [ ]:
import os
from pathlib import Path
import tempfile

# Create a temporary directory for our sample logs
tutorial_dir = Path(tempfile.mkdtemp(prefix="as_scan_tutorial_"))
logs_dir = tutorial_dir / "logs"
logs_dir.mkdir()

print(f"Tutorial directory: {tutorial_dir}")
print(f"Logs directory: {logs_dir}")

In [ ]:
# Create sample log files with various error patterns
sample_logs = {
    "app.log": """
2024-01-15 10:23:45 INFO Application started
2024-01-15 10:23:46 INFO Processing request #1001
2024-01-15 10:23:50 ERROR Database connection failed: Connection timeout
2024-01-15 10:23:51 WARNING Retrying database connection (attempt 1/3)
2024-01-15 10:23:55 ERROR Database connection failed: Connection timeout
2024-01-15 10:24:00 CRITICAL Service unavailable - all retries exhausted
2024-01-15 10:24:05 INFO Application shutting down
""",
    "system.log": """
2024-01-15 08:00:00 INFO System boot complete
2024-01-15 09:15:23 WARNING Disk usage above 80%: /dev/sda1 at 85%
2024-01-15 10:30:45 ERROR Failed to allocate memory: OOM killed process 1234
2024-01-15 10:30:46 ERROR Kernel panic - system halted
""",
    "worker.log": """
2024-01-15 11:00:00 INFO Worker process started (PID: 5678)
2024-01-15 11:15:30 INFO Processing job queue: 42 jobs pending
2024-01-15 11:20:15 ERROR Job #1001 failed: File not found
2024-01-15 11:25:00 INFO Job #1002 completed successfully
2024-01-15 11:30:45 ERROR Job #1003 failed: Permission denied
"""
}

# Write the log files
for filename, content in sample_logs.items():
    log_file = logs_dir / filename
    log_file.write_text(content.strip())
    print(f"Created: {log_file}")

# Verify the files were created
print(f"\nLog files created: {len(list(logs_dir.glob('*.log')))}")

## Step 1: Create Your First Error Catalog

An error catalog is a YAML file that defines:
- What error patterns to look for
- Where to find the files
- What each error means
- How to fix the errors

Let's create a simple catalog that searches for ERROR and CRITICAL messages:

In [ ]:
import yaml
from datetime import datetime

# Define our error catalog
catalog = {
    "version": "1.0.0",
    "schema_version": "1.0.0",
    "metadata": {
        "name": "Tutorial Error Catalog",
        "description": "Simple catalog for tutorial purposes",
        "author": "Tutorial User",
        "created": datetime.now().isoformat(),
        "updated": datetime.now().isoformat()
    },
    "errors": {
        "error_messages": {
            "id": "error_messages",
            "pattern": {
                "type": "regex",
                "pattern": r"ERROR|CRITICAL",
                "flags": ["IGNORECASE"]
            },
            "files": [
                str(logs_dir / "*.log")
            ],
            "meaning": "A critical error was encountered in the system",
            "suggestion": "Review the error context and check system logs",
            "context_lines": 2,
            "next_errors": [],
            "metadata": {
                "severity": "high"
            }
        },
        "oom_killer": {
            "id": "oom_killer",
            "pattern": {
                "type": "literal",
                "pattern": "OOM killed"
            },
            "files": [
                str(logs_dir / "*.log")
            ],
            "meaning": "Process was killed by Out-of-Memory killer",
            "suggestion": "Increase memory allocation or optimize memory usage",
            "context_lines": 3,
            "next_errors": [],
            "metadata": {
                "severity": "critical"
            }
        }
    }
}

# Save the catalog
catalog_path = tutorial_dir / "catalog.yaml"
with open(catalog_path, 'w') as f:
    yaml.dump(catalog, f, default_flow_style=False, sort_keys=False)

print(f"Catalog created: {catalog_path}")
print("\nCatalog contents:")
print(catalog_path.read_text())

## Step 2: Validate the Catalog

Before running a scan, it's good practice to validate your catalog to ensure it's syntactically correct:

In [ ]:
# We can validate programmatically
from src.domain.catalog_io import load_catalog

try:
    loaded_catalog = load_catalog(catalog_path)
    print("✓ Catalog is valid!")
    print(f"  - Found {len(loaded_catalog.errors)} error definitions")
    print(f"  - Catalog name: {loaded_catalog.metadata.name}")
except Exception as e:
    print(f"✗ Catalog validation failed: {e}")

You can also use the CLI to validate:

```bash
as-scan validate catalog.yaml
```

## Step 3: Run Your First Scan

Now let's run the scan! We'll use the Python API for this tutorial, but you can also use the CLI.

In [ ]:
# Run a scan programmatically
from src.orchestration.scanner import run_scan

# Set up output directory
output_dir = tutorial_dir / "results"
output_dir.mkdir()

print(f"Running scan...")
print(f"  Catalog: {catalog_path}")
print(f"  Output: {output_dir}")
print(f"  Cores: 1")
print()

# For demonstration, we'll use a simpler approach
# In practice, you would use: as-scan scan --catalog catalog.yaml --output results
print("Note: In a real scenario, run this command:")
print(f"  as-scan scan --catalog {catalog_path} --output {output_dir} --cores 1")

## Step 4: Understanding the Results

After the scan completes, you'll find several files in the output directory:

- `report.json` - JSON-LD format report with all matches
- `summary.txt` - Quick summary of findings
- `workflow.log` - Snakemake workflow execution log

Let's examine what a typical result looks like:

In [ ]:
# Simulate scan results for demonstration
mock_results = {
    "@context": "https://schema.org/",
    "@type": "Report",
    "name": "Error Scan Report",
    "description": "Scan results from Tutorial Error Catalog",
    "dateCreated": datetime.now().isoformat(),
    "matches": [
        {
            "error_id": "error_messages",
            "file_uri": str(logs_dir / "app.log"),
            "line_number": 3,
            "matched_text": "ERROR Database connection failed: Connection timeout",
            "context_before": [
                "2024-01-15 10:23:45 INFO Application started",
                "2024-01-15 10:23:46 INFO Processing request #1001"
            ],
            "context_after": [
                "2024-01-15 10:23:51 WARNING Retrying database connection (attempt 1/3)",
                "2024-01-15 10:23:55 ERROR Database connection failed: Connection timeout"
            ],
            "meaning": "A critical error was encountered in the system",
            "suggestion": "Review the error context and check system logs"
        },
        {
            "error_id": "oom_killer",
            "file_uri": str(logs_dir / "system.log"),
            "line_number": 3,
            "matched_text": "ERROR Failed to allocate memory: OOM killed process 1234",
            "context_before": [
                "2024-01-15 08:00:00 INFO System boot complete",
                "2024-01-15 09:15:23 WARNING Disk usage above 80%: /dev/sda1 at 85%"
            ],
            "context_after": [
                "2024-01-15 10:30:46 ERROR Kernel panic - system halted"
            ],
            "meaning": "Process was killed by Out-of-Memory killer",
            "suggestion": "Increase memory allocation or optimize memory usage"
        }
    ],
    "summary": {
        "totalMatches": 7,
        "errorTypes": 2,
        "filesScanned": 3
    }
}

print("\n=== Scan Results Summary ===")
print(f"Total matches found: {mock_results['summary']['totalMatches']}")
print(f"Error types detected: {mock_results['summary']['errorTypes']}")
print(f"Files scanned: {mock_results['summary']['filesScanned']}")
print("\n=== Sample Matches ===")
for i, match in enumerate(mock_results['matches'][:2], 1):
    print(f"\nMatch {i}:")
    print(f"  Error ID: {match['error_id']}")
    print(f"  File: {Path(match['file_uri']).name}")
    print(f"  Line: {match['line_number']}")
    print(f"  Text: {match['matched_text'][:60]}...")
    print(f"  Suggestion: {match['suggestion']}")

## Step 5: Exporting Reports

You can export the results in various formats:

### Markdown Export

```bash
as-scan export results/report.json --template markdown --output report.md
```

### HTML Export

```bash
as-scan export results/report.json --template html --output report.html
```

### Plain Text Export

```bash
as-scan export results/report.json --template text --output report.txt
```

## Key Takeaways

In this tutorial, you learned:

1. **Creating Error Catalogs**: Define patterns, files, and metadata
2. **Pattern Types**: Used both `regex` and `literal` patterns
3. **Running Scans**: Execute scans with the CLI or Python API
4. **Understanding Results**: Interpret JSON-LD reports and summaries
5. **Exporting Reports**: Convert results to various formats

## Next Steps

- [Tutorial 2: Pattern Matching Basics](02_pattern_matching.ipynb) - Learn about different pattern types
- [Tutorial 3: Scanning Remote Files](03_remote_files.ipynb) - Access files via SSH, S3, and SFTP
- [Tutorial 4: Using the Railway Pattern](04_railway_pattern.ipynb) - Chain related errors conditionally

## Cleanup

Let's clean up the temporary files we created:

In [ ]:
import shutil

# Clean up tutorial directory
if tutorial_dir.exists():
    shutil.rmtree(tutorial_dir)
    print(f"Cleaned up: {tutorial_dir}")
else:
    print("Tutorial directory already cleaned up")